# BiolytAI — Biomarker Extraction
Model: `Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1` (Qwen3.5-0.8B + LoRA)

Runs over `brief_title`, `primary_outcome`, and `secondary_outcome` for each trial.

In [ ]:
!pip install transformers peft torch accelerate pandas tqdm -q

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import os
import torch
from tqdm.auto import tqdm

tqdm.pandas()

# Load the harmonized CSV from step 1
df = pd.read_csv('output/trials_harmonized_step1.csv')
print(f'Loaded {len(df)} rows')
print('Columns:', df.columns.tolist())

## 1. Load Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'   # underlying base
ADAPTER_ID = 'Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from huggingface_hub import model_info

# Inspect the adapter card to determine the correct base model
try:
    info = model_info(ADAPTER_ID)
    print('Model card tags:', info.tags)
    # The adapter config will tell us the base model
    from huggingface_hub import hf_hub_download
    import json as _json
    adapter_cfg_path = hf_hub_download(ADAPTER_ID, 'adapter_config.json')
    with open(adapter_cfg_path) as f:
        adapter_cfg = _json.load(f)
    BASE_FROM_ADAPTER = adapter_cfg.get('base_model_name_or_path', BASE_MODEL_ID)
    print('Base model declared in adapter config:', BASE_FROM_ADAPTER)
    BASE_MODEL_ID = BASE_FROM_ADAPTER
except Exception as e:
    print(f'Could not read adapter config: {e}')
    print(f'Falling back to: {BASE_MODEL_ID}')

In [ ]:
print(f'Loading tokenizer from {BASE_MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # for batch generation

print(f'Loading base model from {BASE_MODEL_ID} ...')
dtype = torch.float16 if DEVICE == 'cuda' else torch.float32
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=dtype,
    device_map='auto' if DEVICE == 'cuda' else None,
    trust_remote_code=True,
)

print(f'Loading LoRA adapter from {ADAPTER_ID} ...')
model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
model.eval()
if DEVICE == 'cpu':
    model = model.to(DEVICE)

print('Model ready.')

## 2. Prompt Template

In [ ]:
SYSTEM_PROMPT = (
    'You are a biomedical NLP expert. Extract all biomarkers mentioned in the clinical trial text. '
    'Biomarkers include genes, proteins, mutations, expression levels, lab values, imaging findings, '
    'and any measurable biological indicator used for diagnosis, prognosis, or treatment response. '
    'Return ONLY a comma-separated list of biomarkers found. '
    'If no biomarkers are present, return exactly: NONE'
)

def build_prompt(title, primary_outcome, secondary_outcome):
    parts = []
    if isinstance(title, str) and title.strip():
        parts.append(f'Title: {title.strip()}')
    if isinstance(primary_outcome, str) and primary_outcome.strip():
        parts.append(f'Primary outcome: {primary_outcome.strip()}')
    if isinstance(secondary_outcome, str) and secondary_outcome.strip():
        # truncate very long secondary outcomes
        sec = secondary_outcome.strip()[:600]
        parts.append(f'Secondary outcomes: {sec}')
    text = '\n'.join(parts)

    # Chat format (Qwen uses ChatML)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': text},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Quick sanity check
sample_prompt = build_prompt(
    'BRCA1 mutation in HER2+ Breast Cancer',
    'Overall survival | PD-L1 expression',
    'CA-125 levels | CTCs'
)
print(sample_prompt)

## 3. Batched Inference

In [ ]:
# Batch size: 16 on GPU, 4 on CPU
BATCH_SIZE = 16 if DEVICE == 'cuda' else 4
MAX_NEW_TOKENS = 128

def parse_biomarkers(raw_output):
    """Parse model output -> deduplicated list of biomarkers."""
    if not isinstance(raw_output, str):
        return []
    # Strip everything before the last assistant turn
    if '<|im_start|>assistant' in raw_output:
        raw_output = raw_output.split('<|im_start|>assistant')[-1]
    raw_output = raw_output.replace('<|im_end|>', '').strip()
    if raw_output.upper().startswith('NONE') or raw_output.strip() == '':
        return []
    # Split on comma or semicolon
    parts = re.split(r'[,;]', raw_output)
    biomarkers = []
    seen = set()
    for p in parts:
        p = p.strip().strip('.')
        p = re.sub(r'^[-•*\d\.]+\s*', '', p)  # remove bullet/numbering
        p_lower = p.lower()
        if p and len(p) >= 2 and p_lower not in seen and p_lower != 'none':
            seen.add(p_lower)
            biomarkers.append(p)
    return biomarkers

def run_batch(prompts):
    """Run inference on a batch of prompt strings, return list of raw outputs."""
    inputs = tokenizer(
        prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,       # greedy — deterministic, faster
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    input_len = inputs['input_ids'].shape[1]
    decoded = tokenizer.batch_decode(
        outputs[:, input_len:],
        skip_special_tokens=True
    )
    return decoded

In [ ]:
import time

all_biomarkers = []
rows = df.to_dict('records')
n = len(rows)

t0 = time.time()

for batch_start in tqdm(range(0, n, BATCH_SIZE), desc='Biomarker extraction'):
    batch = rows[batch_start: batch_start + BATCH_SIZE]
    prompts = [
        build_prompt(r.get('brief_title'), r.get('primary_outcome'), r.get('secondary_outcome'))
        for r in batch
    ]
    raw_outputs = run_batch(prompts)
    for raw in raw_outputs:
        biomarkers = parse_biomarkers(raw)
        all_biomarkers.append(biomarkers)

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s ({elapsed/n*1000:.1f}ms/trial, batch_size={BATCH_SIZE})')

## 4. Attach results & Quality Metrics

In [ ]:
df['biomarkers'] = [', '.join(b) if b else '' for b in all_biomarkers]

# Quality metrics
n_extracted = (df['biomarkers'] != '').sum()
n_empty = (df['biomarkers'] == '').sum()
# Trials with no text at all (no title, no primary outcome)
no_text = df[['brief_title', 'primary_outcome', 'secondary_outcome']].isnull().all(axis=1).sum()

total_biomarkers_found = sum(len(b) for b in all_biomarkers)
avg_per_trial = total_biomarkers_found / n_extracted if n_extracted else 0

print(f'Total trials:                  {n}')
print(f'Trials with biomarkers:        {n_extracted} ({n_extracted/n*100:.1f}%)')
print(f'Trials with no biomarkers:     {n_empty} ({n_empty/n*100:.1f}%)')
print(f'Trials with no text at all:    {no_text}')
print(f'Total biomarker mentions:      {total_biomarkers_found}')
print(f'Avg biomarkers per trial:      {avg_per_trial:.2f}')

# Top biomarkers
from collections import Counter
all_flat = [b for bl in all_biomarkers for b in bl]
top = Counter(all_flat).most_common(20)
print('\nTop 20 biomarkers:')
for bm, count in top:
    print(f'  {bm}: {count}')

In [ ]:
# Save biomarker metrics
biomarker_metrics = {
    'total_trials': int(n),
    'trials_with_biomarkers': int(n_extracted),
    'extraction_rate_pct': round(n_extracted / n * 100, 1),
    'trials_no_biomarkers': int(n_empty),
    'trials_no_input_text': int(no_text),
    'total_biomarker_mentions': int(total_biomarkers_found),
    'avg_biomarkers_per_positive_trial': round(avg_per_trial, 2),
    'top_20_biomarkers': [{'biomarker': b, 'count': c} for b, c in top],
    'inference_time_seconds': round(elapsed, 1),
    'batch_size': BATCH_SIZE,
    'device': DEVICE,
}

with open('output/biomarker_metrics.json', 'w') as f:
    json.dump(biomarker_metrics, f, indent=2)

print('Biomarker metrics saved.')

## 5. Save Final Output

In [ ]:
# Reorder columns: originals + 4 clean columns + biomarkers
ORIG_COLS = ['nct_id', 'brief_title', 'official_title', 'overall_status', 'start_date',
             'sponsor', 'condition', 'phase', 'intervention_name', 'intervention_type',
             'primary_outcome', 'secondary_outcome', 'country', 'study_type', 'enrollment']
CLEAN_COLS = ['intervention_name_clean', 'condition_clean', 'sponsor_clean', 'country_clean', 'biomarkers']

final_cols = ORIG_COLS + CLEAN_COLS
df_final = df[[c for c in final_cols if c in df.columns]]

df_final.to_csv('output/trials_cleaned_final.csv', index=False)
print(f'Final CSV saved: {len(df_final)} rows, {len(df_final.columns)} columns')
print('Columns:', df_final.columns.tolist())

In [ ]:
# Quick sample to verify
sample = df_final[df_final['biomarkers'] != ''][['nct_id', 'brief_title', 'biomarkers']].head(10)
for _, row in sample.iterrows():
    print(f"{row['nct_id']}: {row['brief_title'][:60]}...")
    print(f"  Biomarkers: {row['biomarkers']}")
    print()

## 6. Edge Cases Summary

In [ ]:
# Flag edge cases
edge_cases = []

# Trials where primary_outcome is null (model had only title to work from)
no_primary = df_final[df_final['primary_outcome'].isnull() & (df_final['biomarkers'] != '')]
edge_cases.append({'type': 'biomarker_from_title_only', 'count': len(no_primary)})

# Trials with very long biomarker lists (> 10 items, possible hallucination)
long_lists = df_final[df_final['biomarkers'].apply(lambda x: len(x.split(',')) if isinstance(x, str) and x else 0) > 10]
edge_cases.append({'type': 'long_biomarker_list_gt_10', 'count': len(long_lists)})

# Non-drug/non-biomarker trials (OTHER intervention type) that got biomarkers extracted
non_drug_with_biomarkers = df_final[
    (df_final['intervention_type'] == 'OTHER') & (df_final['biomarkers'] != '')
]
edge_cases.append({'type': 'other_intervention_with_biomarkers', 'count': len(non_drug_with_biomarkers)})

for e in edge_cases:
    print(f"{e['type']}: {e['count']} trials")

with open('output/edge_cases.json', 'w') as f:
    json.dump(edge_cases, f, indent=2)

print('\nAll output files:')
for fname in sorted(os.listdir('output')):
    size = os.path.getsize(f'output/{fname}')
    print(f'  output/{fname}  ({size/1024:.1f} KB)')